In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/achiver07/data-project/acts_train.jsonl
/kaggle/input/datasets/achiver07/data-project/acts_val.jsonl
/kaggle/input/datasets/achiver07/data-project/sc_train.jsonl
/kaggle/input/datasets/achiver07/data-project/sc_val.jsonl
/kaggle/input/datasets/achiver07/data-project/test.json
/kaggle/input/datasets/achiver07/data-project/tribunals_train.jsonl
/kaggle/input/datasets/achiver07/data-project/tribunals_val.jsonl
/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/acts_train.bin
/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/acts_val.bin
/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/sc_train.bin
/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/sc_val.bin
/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/test.json
/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/tokenizer.json
/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/tribunals_train_f.bin
/kaggle/input/datasets/achiver07/phase1-tokens-v40k-cl

In [2]:
import os
from kaggle_secrets import UserSecretsClient
from huggingface_hub import snapshot_download

# Read token securely from Kaggle Secrets
secrets = UserSecretsClient()
hf_token = secrets.get_secret("HF_TOKEN")

REPO_ID = "Exploration-Lab/CS-781-Capstone"
LOCAL_DIR = "/kaggle/working/Phase_4_CJPE"

# Download only Phase 4 files
path = snapshot_download(
    repo_id=REPO_ID,
    repo_type="dataset",
    token=hf_token,
    allow_patterns=[
        "Phase_4_CJPE/**"
    ],
    local_dir=LOCAL_DIR,
)

print("Download completed!")
print("Dataset location:", path)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Download completed!
Dataset location: /kaggle/working/Phase_4_CJPE


In [3]:
for root, dirs, files in os.walk(LOCAL_DIR):
    for file in files:
        print(os.path.join(root, file))
        

/kaggle/working/Phase_4_CJPE/.cache/huggingface/.gitignore
/kaggle/working/Phase_4_CJPE/.cache/huggingface/CACHEDIR.TAG
/kaggle/working/Phase_4_CJPE/.cache/huggingface/download/Phase_4_CJPE/Test/test.json.lock
/kaggle/working/Phase_4_CJPE/.cache/huggingface/download/Phase_4_CJPE/Test/test.json.metadata
/kaggle/working/Phase_4_CJPE/.cache/huggingface/download/Phase_4_CJPE/Train/dev.json.metadata
/kaggle/working/Phase_4_CJPE/.cache/huggingface/download/Phase_4_CJPE/Train/train.json.lock
/kaggle/working/Phase_4_CJPE/.cache/huggingface/download/Phase_4_CJPE/Train/train.json.metadata
/kaggle/working/Phase_4_CJPE/.cache/huggingface/download/Phase_4_CJPE/Train/dev.json.lock
/kaggle/working/Phase_4_CJPE/.cache/huggingface/trees/93b0059be756c6c461f3a225de25aabda7d5f84a.json
/kaggle/working/Phase_4_CJPE/Phase_4_CJPE/Test/test.json
/kaggle/working/Phase_4_CJPE/Phase_4_CJPE/Train/dev.json
/kaggle/working/Phase_4_CJPE/Phase_4_CJPE/Train/train.json


In [4]:
# =============================================================================
# PHASE 4 — EXPERIMENT 2 (UPGRADED): PHASE-1 ARCH3 + HIGH-RANK ADALORA
# ONE-CELL KAGGLE TRAINING SCRIPT — FULL LABELLED CORPUS, NO EVALUATION
# =============================================================================
# KEY UPGRADES FOR 0.70+ SCORE & FASTER T4 THROUGHPUT:
#   1. 50% More Context (MAX_ZONE_LEN = 384 -> 1,536 tokens/case). Arch3 uses
#      Sliding Window Attention (W=256), so longer zones scale linearly.
#   2. Double Hardware Batch Size (BATCH_DOCS = 8, GRAD_ACCUM = 2 -> eff. 16)
#      with cudnn.benchmark and pinned multi-worker loading for faster epochs.
#   3. 2 Full Epochs with OneCycleLR (Warmup + Cosine Annealing) instead of a
#      static 1-epoch learning rate.
#   4. Higher-Capacity AdaLoRA (init_r=16 -> target_r=8, alpha=32) + unfrozen
#      backbone final_norm for cleaner feature scaling.
#   5. Class-Balanced Focal Loss (gamma=2.0) to focus gradients on hard legal
#      cases without distorting logits via raw frequency weights.
#   6. 6x Rich Context Fusion + 5-Sample Multi-Sample Dropout Classifier Head.
#   7. Prior-Calibrated Decision Threshold + Temperature-Scaled Logit Blending
#      (uses only training label prior + test probability distribution; 0 dev/test leakage).
#
# PATCHES APPLIED (see "Fixes" section below for details):
#   A. FIX: AdaLoRA `update_and_allocate` was being called AFTER `zero_grad`, so
#      p.grad was None -> "Parameter * NoneType" crash. It now runs BEFORE
#      `zero_grad`, matching PEFT's reference loop.
#   B. FIX: fp16 GradScaler overflow steps poison AdaLoRA's exponential moving
#      averages. `update_and_allocate` is now skipped on non-finite grad_norm
#      until the importance phase ends.
#   C. FIX: orthogonal regularization (`orth_reg_weight=0.1`) was silently dead
#      because we bypass PEFT's AdaLoraModel.forward. It is now applied manually.
#
# NOTES (not changed, but worth knowing):
#   * Runtime: the 1.99 s/it on iteration 1 includes CUDA/cuDNN warmup. Steady
#     state ~2 s/it => ~5.5h for 2 epochs, which fits Kaggle but eats quota.
#     If it is much slower, drop MAX_ZONE_LEN to 256 or set EPOCHS = 1.
#   * Memory: 32 x 384 sequences + no checkpointing is tight on a 14.5 GiB T4.
#     On OOM set USE_GRAD_CHECKPOINTING=True or BATCH_DOCS=4, GRAD_ACCUM=4.
#   * No validation signal: dev is merged into training, so nothing is scored
#     before submission. The 0.70 target is a hope, not a measured result. If
#     you have attempts to spare, hold out ~2k dev examples for a 1-epoch F1.
#   * Threshold calibration is transductive (uses the test prob distribution)
#     and assumes test class balance matches training. Clip to [0.35, 0.65]
#     limits damage if that assumption fails.
# =============================================================================

import os, sys, json, time, math, random, gc, zipfile, shutil, warnings
from pathlib import Path
from collections import Counter
from types import SimpleNamespace

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from torch.utils.checkpoint import checkpoint
from tokenizers import Tokenizer
from tqdm.auto import tqdm

# Install PEFT if missing (software library only, no pretrained weights)
try:
    import peft
    from peft import AdaLoraConfig, get_peft_model, PeftModel
except Exception:
    import subprocess
    print("PEFT not found; installing peft==0.17.1 (library only)...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "peft==0.17.1"])
    import peft
    from peft import AdaLoraConfig, get_peft_model, PeftModel

assert torch.cuda.is_available(), "Select a GPU accelerator in Kaggle Settings before running."
DEVICE = torch.device("cuda")
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ======================= PATHS & HYPERPARAMETERS =======================
DATA_ROOT = Path("/kaggle/working/Phase_4_CJPE/Phase_4_CJPE")
TRAIN_PATH = DATA_ROOT / "Train" / "train.json"
DEV_PATH   = DATA_ROOT / "Train" / "dev.json"
TEST_PATH  = DATA_ROOT / "Test" / "test.json"

OUT_DIR = Path("/kaggle/working/Phase4_Submissions/Exp2_Arch3_AdaLoRA_Improved")
OUT_DIR.mkdir(parents=True, exist_ok=True)
ADAPTER_DIR = OUT_DIR / "adalora_adapter"
HEAD_PATH = OUT_DIR / "classifier_head.pt"
FINAL_NORM_PATH = OUT_DIR / "final_norm.pt"
SUBMISSION_PATH = OUT_DIR / "prediction.json"
ZIP_PATH = OUT_DIR / "submission.zip"
RESULTS_PATH = OUT_DIR / "training_metadata.json"

MAX_TRAIN_DOCS = None          # None = full corpus (train + dev merged)
EPOCHS = 2                     # 2 epochs with cosine schedule for full convergence
BATCH_DOCS = 8                 # 8 docs x 4 zones = 32 sequences/step (fits easily in 14.5 GiB T4)
GRAD_ACCUM_STEPS = 2           # Effective batch size = 16 documents per optimizer step
MAX_ZONE_LEN = 384             # 382 content tokens + BOS/EOS = 1,536 tokens per legal case
USE_GRAD_CHECKPOINTING = False # Keep False for maximum T4 speed; set True only if OOM
CALIBRATE_THRESHOLD_TO_PRIOR = True # Match test positive rate to training corpus prior

print("PyTorch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0))
print("GPU memory (GiB):", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
print(f"Speed & Context Profile: epochs={EPOCHS}, batch_docs={BATCH_DOCS} (eff={BATCH_DOCS*GRAD_ACCUM_STEPS}), zone_len={MAX_ZONE_LEN}")

# ======================= PHASE-1 BACKBONE LOADER =======================
VOCAB = 40000
EXPECTED_PARAMS = {"arch1": 88_210_944, "arch2": 129_957_888, "arch3": 242_501_632}

DEFAULT_CKPTS = {
    "arch1": "/kaggle/input/models/achiver7/arch1-modeldecoder/transformers/default/1/run_arch1_real/best.pt",
    "arch2": "/kaggle/input/models/achiver7/arch2-model/transformers/default/1/run_arch2_preln_efficient/best.pt",
    "arch3": "/kaggle/input/datasets/achiver7/arch3-model/run_arch3_swa_v2/best.pt",
}

_ARCH3_SRC = r'''def sinusoidal_pos_emb(T, d_model, device):
    pos = torch.arange(T, device=device, dtype=torch.float32).unsqueeze(1)
    div = torch.exp(
        torch.arange(0, d_model, 2, device=device, dtype=torch.float32)
        * (-math.log(10000.0) / d_model)
    )
    pe = torch.zeros(T, d_model, device=device, dtype=torch.float32)
    pe[:, 0::2] = torch.sin(pos * div)
    pe[:, 1::2] = torch.cos(pos * div)
    return pe.unsqueeze(0)


class SlidingWindowCausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, block_size, window_size):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model = d_model
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.window_size = window_size

        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)

        idx = torch.arange(block_size)
        causal = idx[None, :] <= idx[:, None]
        window = (idx[:, None] - idx[None, :]) < window_size
        mask = causal & window

        self.register_buffer(
            "swa_mask",
            mask.view(1, 1, block_size, block_size),
            persistent=False,
        )

    def forward(self, x):
        B, T, C = x.shape
        q = self.q_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)

        attn_mask = self.swa_mask[:, :, :T, :T]
        y = F.scaled_dot_product_attention(
            q, k, v, attn_mask=attn_mask, dropout_p=0.0, is_causal=False
        )
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.out_proj(y)


class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)

    def forward(self, x):
        return self.fc2(F.gelu(self.fc1(x)))


class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, block_size, window_size):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attn = SlidingWindowCausalSelfAttention(
            d_model=d_model, n_heads=n_heads, block_size=block_size, window_size=window_size
        )
        self.norm2 = nn.LayerNorm(d_model)
        self.mlp = FeedForward(d_model, d_ff)

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x


class LegalDecoderLM(nn.Module):
    def __init__(self, vocab_size, block_size, d_model, n_layers, n_heads, d_ff, window_size):
        super().__init__()
        self.vocab_size = vocab_size
        self.block_size = block_size
        self.d_model = d_model
        self.n_layers = n_layers

        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.embed_scale = math.sqrt(d_model)

        pos = sinusoidal_pos_emb(block_size, d_model, device="cpu")
        self.register_buffer("pos_emb", pos, persistent=False)

        self.blocks = nn.ModuleList([
            TransformerBlock(
                d_model=d_model, n_heads=n_heads, d_ff=d_ff, block_size=block_size, window_size=window_size
            )
            for _ in range(n_layers)
        ])

        self.final_norm = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.token_emb.weight

        self.apply(self._init_weights)
        for block in self.blocks:
            block.attn.out_proj.weight.data.mul_(1.0 / math.sqrt(2 * self.n_layers))
            block.mlp.fc2.weight.data.mul_(1.0 / math.sqrt(2 * self.n_layers))

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
        elif isinstance(module, nn.LayerNorm):
            nn.init.ones_(module.weight)
            nn.init.zeros_(module.bias)

    def forward(self, x):
        B, T = x.shape
        assert T <= self.block_size
        tok = self.token_emb(x) * self.embed_scale
        pos = self.pos_emb[:, :T, :].to(tok.dtype)
        h = tok + pos
        for block in self.blocks:
            h = block(h)
        h = self.final_norm(h)
        return self.lm_head(h)


def build_model(vocab_size, block_size, window_size=256):
    return LegalDecoderLM(
        vocab_size=vocab_size, block_size=block_size, d_model=1024,
        n_layers=16, n_heads=16, d_ff=4096, window_size=window_size,
    )
'''

def _namespace(src, name):
    ns = {"math": math, "torch": torch, "nn": nn, "F": F, "__name__": f"phase1_{name}"}
    exec(compile(src, f"<phase1 {name} definition>", "exec"), ns)
    return ns

def _strip_compile_prefix(sd):
    return {(k[len("_orig_mod."):] if k.startswith("_orig_mod.") else k): v for k, v in sd.items()}

def load_backbone(name, ckpt_path=None, device="cuda"):
    assert name == "arch3", "This pipeline is specialized for Phase-1 Arch3."
    ckpt_path = ckpt_path or os.environ.get("CKPT_ARCH3") or DEFAULT_CKPTS["arch3"]
    ns = _namespace(_ARCH3_SRC, name)
    model, max_len, d_model = ns["build_model"](VOCAB, 1024, 256), 1024, 1024

    def hidden(ids):
        B, T = ids.shape
        assert T <= max_len
        h = model.token_emb(ids) * model.embed_scale + model.pos_emb[:, :T, :].to(model.token_emb.weight.dtype)
        for block in model.blocks:
            h = block(h)
        return model.final_norm(h)

    ck = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    state = ck["model"] if isinstance(ck, dict) and "model" in ck else ck
    model.load_state_dict(_strip_compile_prefix(state), strict=True)
    n_params = sum(p.numel() for p in model.parameters())
    meta = {
        "name": name, "ckpt": ckpt_path, "n_params": n_params,
        "step": ck.get("step") if isinstance(ck, dict) else None,
        "best": ck.get("best") if isinstance(ck, dict) else None,
    }
    model = model.to(device).eval()
    return SimpleNamespace(model=model, hidden=hidden, max_len=max_len, d_model=d_model, meta=meta)

# ======================= LOCATE FILES & LOAD DATA =======================
INPUT_ROOT = Path("/kaggle/input")

arch3_default = Path(os.environ.get("CKPT_ARCH3", DEFAULT_CKPTS["arch3"]))
if arch3_default.is_file():
    ARCH3_CKPT = arch3_default
else:
    candidates = [p for p in INPUT_ROOT.rglob("best.pt") if "arch3" in str(p).lower()]
    candidates = sorted(candidates, key=lambda p: ("arch3-model" not in str(p).lower(), len(str(p))))
    if not candidates:
        raise FileNotFoundError(f"Arch3 Phase 1 best.pt not found. Expected default: {DEFAULT_CKPTS['arch3']}")
    ARCH3_CKPT = candidates[0]

preferred_tokenizer = Path("/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/tokenizer.json")
if preferred_tokenizer.is_file():
    TOKENIZER_PATH = preferred_tokenizer
else:
    toks = sorted(INPUT_ROOT.rglob("tokenizer.json"), key=lambda p: ("phase1-tokens-v40k-clean1" not in str(p).lower(), len(str(p))))
    if not toks:
        raise FileNotFoundError("Phase 1 tokenizer.json not found under /kaggle/input.")
    TOKENIZER_PATH = toks[0]

if not all(p.is_file() for p in (TRAIN_PATH, DEV_PATH, TEST_PATH)):
    print("Phase 4 files not found in /kaggle/working; attempting Hugging Face dataset download...")
    from kaggle_secrets import UserSecretsClient
    from huggingface_hub import snapshot_download
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    snapshot_download(
        repo_id="Exploration-Lab/CS-781-Capstone",
        repo_type="dataset",
        token=hf_token,
        allow_patterns=["Phase_4_CJPE/**"],
        local_dir="/kaggle/working/Phase_4_CJPE",
    )

def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

train_split_records = load_json(TRAIN_PATH)
dev_split_records   = load_json(DEV_PATH)
test_raw            = load_json(TEST_PATH)
test_records        = [{"id": row["id"], "text": row.get("text", "")} for row in test_raw]
del test_raw

test_ids = [str(r["id"]) for r in test_records]
assert len(set(test_ids)) == len(test_ids), "Duplicate IDs in test split!"

SOURCE_TRAIN_COUNT = len(train_split_records)
SOURCE_DEV_COUNT   = len(dev_split_records)
corpus_records     = train_split_records + dev_split_records

if MAX_TRAIN_DOCS is not None:
    rng = np.random.default_rng(SEED)
    n_keep = min(int(MAX_TRAIN_DOCS), len(corpus_records))
    chosen = np.sort(rng.choice(len(corpus_records), size=n_keep, replace=False))
    corpus_records = [corpus_records[i] for i in chosen]

train_records = corpus_records
y_train = np.asarray([int(r["label"]) for r in train_records], dtype=np.int64)
TRAIN_POS_PRIOR = float(y_train.mean())

print(f"Full labelled training corpus: {len(train_records):,} (Pos prior: {TRAIN_POS_PRIOR:.4f})")
print(f"Test examples (id/text only):  {len(test_records):,}")

# ======================= 4-ZONE TOKENIZATION (384 LEN) =======================
tokenizer = Tokenizer.from_file(str(TOKENIZER_PATH))
BOS_ID, EOS_ID, PAD_ID = 1, 2, 0
N_ZONES = 4

def select_token_zones(full_token_ids, zone_tokens):
    """Select 4 strategic zones: Case Facts (0%), Issue/Arguments (30%), Ratio (68%), Final Holding (100%)."""
    ids = list(map(int, full_token_ids))
    n = len(ids)
    if n == 0:
        return [[], [], [], []]

    if n <= N_ZONES * zone_tokens:
        boundaries = np.linspace(0, n, N_ZONES + 1, dtype=int)
        return [ids[boundaries[i]:boundaries[i + 1]] for i in range(N_ZONES)]

    last_start = n - zone_tokens
    # Biased slightly toward the second half of the judgment where legal reasoning & holdings concentrate
    starts = [
        0,
        int(round(last_start * 0.30)),
        int(round(last_start * 0.68)),
        last_start,
    ]
    starts = [max(0, min(s, last_start)) for s in starts]
    starts[-1] = last_start
    return [ids[s:s + zone_tokens] for s in starts]

def encode_records(records, split_name, batch_docs=128):
    n = len(records)
    content_len = MAX_ZONE_LEN - 2
    token_ids = np.zeros((n, N_ZONES, MAX_ZONE_LEN), dtype=np.int32)
    lengths   = np.zeros((n, N_ZONES), dtype=np.int16)

    tokenizer.no_truncation()
    for start in tqdm(range(0, n, batch_docs), desc=f"Tokenizing {split_name}"):
        batch = records[start:start + batch_docs]
        texts = [str(row.get("text") or "") for row in batch]
        encodings = tokenizer.encode_batch(texts, add_special_tokens=False)
        for offset, enc in enumerate(encodings):
            doc_i = start + offset
            zones = select_token_zones(enc.ids, content_len)
            for zone_i, zone in enumerate(zones):
                content = [int(t) for t in zone[:content_len]]
                seq = [BOS_ID] + content + [EOS_ID]
                seq_len = min(len(seq), MAX_ZONE_LEN)
                token_ids[doc_i, zone_i, :seq_len] = seq[:seq_len]
                lengths[doc_i, zone_i] = seq_len

    assert token_ids.min() >= 0 and token_ids.max() < VOCAB
    return token_ids, lengths

X_train, L_train = encode_records(train_records, "full labelled corpus")
X_test, L_test   = encode_records(test_records, "test inputs (text only)")
del train_split_records, dev_split_records, corpus_records, train_records
gc.collect()

print("Encoded shapes:", X_train.shape, X_test.shape)
print("Mean zone lengths (train):", np.round(L_train.mean(axis=0), 1).tolist())

# ======================= LOAD ARCH3 & INJECT ADALORA =======================
bb3 = load_backbone("arch3", ckpt_path=str(ARCH3_CKPT), device=str(DEVICE))
assert bb3.meta["n_params"] == EXPECTED_PARAMS["arch3"]

TARGET_SUFFIXES = ("q_proj", "k_proj", "v_proj", "out_proj", "fc1", "fc2")
target_names = [
    name for name, module in bb3.model.named_modules()
    if isinstance(module, nn.Linear) and name.split(".")[-1] in TARGET_SUFFIXES
]
assert len(target_names) == 16 * 6

train_loader = DataLoader(
    TensorDataset(
        torch.from_numpy(X_train).long(),
        torch.from_numpy(L_train).long(),
        torch.from_numpy(y_train).long(),
    ),
    batch_size=BATCH_DOCS,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True,
)

steps_per_epoch = math.ceil(len(train_loader) / GRAD_ACCUM_STEPS)
total_steps = EPOCHS * steps_per_epoch
TINIT  = max(1, int(total_steps * 0.08))
TFINAL = max(1, int(total_steps * 0.15))

# Upgraded AdaLoRA capacity: init_r=16 -> target_r=8 with alpha=32
adalora_config = AdaLoraConfig(
    init_r=16,
    target_r=8,
    tinit=TINIT,
    tfinal=TFINAL,
    deltaT=20,
    beta1=0.85,
    beta2=0.85,
    lora_alpha=32,
    lora_dropout=0.05,
    orth_reg_weight=0.1,
    total_step=total_steps,
    target_modules=list(TARGET_SUFFIXES),
    task_type=None,
    bias="none",
)

adalora_model = get_peft_model(bb3.model, adalora_config)

# Unfreeze final_norm so the backbone output scale adapts smoothly to the classifier head
for p in bb3.model.final_norm.parameters():
    p.requires_grad = True

adalora_model.train()
adalora_model.print_trainable_parameters()

# ======================= ENHANCED MULTI-SAMPLE DROPOUT HEAD =======================
class EnhancedCJPEZoneClassifier(nn.Module):
    """
    Combines gated attention pooling over all 4 zones with explicit zone features
    (Facts, Ratio, Holding, and progression differences) + 5-way Multi-Sample Dropout.
    """
    def __init__(self, d_model=1024, dropout=0.25):
        super().__init__()
        self.token_gate = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, 128),
            nn.GELU(),
            nn.Linear(128, 1),
        )
        self.zone_score = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, 256),
            nn.GELU(),
            nn.Linear(256, 1),
        )
        # 6 fused vectors of size d_model -> 6 * 1024 = 6144
        fused_dim = d_model * 6
        self.pre_proj = nn.Sequential(
            nn.LayerNorm(fused_dim),
            nn.Linear(fused_dim, 768),
            nn.GELU(),
            nn.LayerNorm(768),
        )
        # Multi-sample dropout branches for faster convergence & better generalization
        self.dropouts = nn.ModuleList([nn.Dropout(p) for p in (0.10, 0.20, 0.30, 0.40, 0.50)])
        self.out_proj = nn.Linear(768, 2)

    def pool_tokens(self, h, flat_lengths):
        # h: [B*Z, T, D], flat_lengths: [B*Z]
        BZ, T, D = h.shape
        positions = torch.arange(T, device=h.device).unsqueeze(0)
        valid_mask = positions < flat_lengths.unsqueeze(1) # [B*Z, T]

        # 1) Masked Mean Pool
        mask_f = valid_mask.unsqueeze(-1).to(h.dtype)
        mean_pool = (h * mask_f).sum(dim=1) / mask_f.sum(dim=1).clamp(min=1.0)

        # 2) Token-level Attention Pool inside each zone
        gate_logits = self.token_gate(h).squeeze(-1) # [B*Z, T]
        gate_logits = gate_logits.masked_fill(~valid_mask, float("-inf"))
        gate_weights = torch.softmax(gate_logits, dim=-1).unsqueeze(-1).to(h.dtype)
        attn_pool = (h * gate_weights).sum(dim=1)

        # 3) Causal End-Token State (summarizes full sliding window at end of zone)
        row = torch.arange(BZ, device=h.device)
        end_pool = h[row, flat_lengths - 1]

        return (0.45 * attn_pool + 0.35 * end_pool + 0.20 * mean_pool)

    def forward(self, zone_reps):
        # zone_reps: [B, 4, D]
        weights = torch.softmax(self.zone_score(zone_reps).squeeze(-1), dim=1)
        pooled  = torch.sum(zone_reps * weights.unsqueeze(-1), dim=1)

        facts   = zone_reps[:, 0]
        ratio   = zone_reps[:, 2]
        holding = zone_reps[:, 3]

        fused = torch.cat([
            pooled,
            facts,
            ratio,
            holding,
            holding - ratio,
            holding - facts,
        ], dim=-1)

        hidden = self.pre_proj(fused)
        logits = torch.stack([self.out_proj(drop(hidden)) for drop in self.dropouts], dim=0).mean(dim=0)
        return logits, weights

classifier = EnhancedCJPEZoneClassifier(d_model=bb3.d_model).to(DEVICE)

def arch3_hidden(ids):
    model = bb3.model
    B, T = ids.shape
    h = model.token_emb(ids) * model.embed_scale + model.pos_emb[:, :T, :].to(model.token_emb.weight.dtype)
    for block in model.blocks:
        if USE_GRAD_CHECKPOINTING and model.training and torch.is_grad_enabled():
            h = checkpoint(block, h, use_reentrant=False)
        else:
            h = block(h)
    return model.final_norm(h)

def forward_document_batch(input_ids, lengths):
    B, Z, T = input_ids.shape
    flat_ids = input_ids.reshape(B * Z, T)
    flat_lengths = lengths.reshape(B * Z).clamp(min=1, max=T)
    h = arch3_hidden(flat_ids)
    zone_vecs = classifier.pool_tokens(h, flat_lengths).reshape(B, Z, -1)
    return classifier(zone_vecs)

# ======================= CLASS-BALANCED FOCAL LOSS =======================
class ClassBalancedFocalLoss(nn.Module):
    """
    Down-weights easy examples via (1 - pt)^gamma while applying mild inverse-prior
    alpha weighting so the model neither collapses to the majority class nor skews
    raw probability calibration.
    """
    def __init__(self, class_counts, gamma=2.0, smoothing=0.02):
        super().__init__()
        counts = np.asarray(class_counts, dtype=np.float64)
        # Mild fourth-root inverse weighting prevents extreme probability distortion
        weights = (counts.sum() / np.maximum(counts, 1.0)) ** 0.25
        weights = weights / weights.mean()
        self.register_buffer("alpha", torch.tensor(weights, dtype=torch.float32))
        self.gamma = gamma
        self.smoothing = smoothing

    def forward(self, logits, targets):
        log_probs = F.log_softmax(logits.float(), dim=-1)
        probs = torch.exp(log_probs)

        targets_one_hot = F.one_hot(targets, num_classes=2).float()
        if self.smoothing > 0:
            targets_one_hot = targets_one_hot * (1.0 - self.smoothing) + 0.5 * self.smoothing

        pt = (probs * targets_one_hot).sum(dim=-1).clamp(min=1e-6, max=1.0 - 1e-6)
        focal_Term = (1.0 - pt) ** self.gamma
        ce_term = -(targets_one_hot * log_probs).sum(dim=-1)
        alpha_t = self.alpha[targets]

        return (alpha_t * focal_Term * ce_term).mean()

# ======================= ADALORA ORTHOGONAL REGULARIZATION =======================
def adalora_orth_reg(model, weight=0.1):
    """
    Re-implements PEFT's internal AdaLoRA orthogonality penalty.

    PEFT applies `orth_reg_weight * reg` inside AdaLoraModel.forward, but this
    script bypasses that forward (it calls model.token_emb / model.blocks
    directly), so the penalty was silently dropped. We add it back here.

    The penalty pushes each LoRA A/B pair toward an orthonormal basis, which
    stabilizes rank-importance estimates and makes pruning less destructive.
    """
    reg = torch.zeros((), device=next(model.parameters()).device, dtype=torch.float32)
    n = 0
    with torch.autocast("cuda", enabled=False):
        for name, p in model.named_parameters():
            if ("lora_A" in name or "lora_B" in name) and p.numel() > 0:
                pf = p.float()
                cov = pf @ pf.T if "lora_A" in name else pf.T @ pf
                eye = torch.eye(cov.size(0), device=cov.device, dtype=cov.dtype)
                reg = reg + torch.norm(cov - eye, p="fro")
                n += 1
    return weight * reg / max(n, 1)

# Smoke test
probe_ids  = torch.from_numpy(X_train[:1]).long().to(DEVICE)
probe_lens = torch.from_numpy(L_train[:1]).long().to(DEVICE)
adalora_model.eval(); classifier.eval()
with torch.no_grad(), torch.amp.autocast("cuda", dtype=torch.float16):
    probe_logits, probe_w = forward_document_batch(probe_ids, probe_lens)
assert probe_logits.shape == (1, 2)
print("Forward smoke test passed; logits shape:", tuple(probe_logits.shape))
del probe_ids, probe_lens, probe_logits, probe_w

# ======================= OPTIMIZER & COSINE SCHEDULER =======================
adapter_params = [p for p in adalora_model.parameters() if p.requires_grad]
head_params    = list(classifier.parameters())
trainable_params = adapter_params + head_params

n_base_total = int(bb3.meta["n_params"])
n_trainable  = sum(p.numel() for p in trainable_params)

optimizer = torch.optim.AdamW(
    [
        {"params": adapter_params, "lr": 1.2e-4, "weight_decay": 0.01},
        {"params": head_params,    "lr": 3.5e-4, "weight_decay": 0.02},
    ]
)

scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=[1.2e-4, 3.5e-4],
    total_steps=total_steps,
    pct_start=0.08,
    anneal_strategy="cos",
    div_factor=10.0,
    final_div_factor=100.0,
)

counts = np.bincount(y_train, minlength=2)
criterion = ClassBalancedFocalLoss(counts, gamma=2.0, smoothing=0.02).to(DEVICE)
scaler = torch.amp.GradScaler("cuda", enabled=True)

print(f"Total optimizer steps={total_steps:,} | tinit={TINIT:,} | tfinal={TFINAL:,}")
print(f"Trainable params (AdaLoRA + final_norm + Head)={n_trainable:,} ({100*n_trainable/(n_base_total+n_trainable):.3f}%)")

# ======================= FULL-CORPUS TRAINING LOOP =======================
torch.cuda.reset_peak_memory_stats()
start_time = time.time()
epoch_losses = []
global_step = 0

if ADAPTER_DIR.exists():
    shutil.rmtree(ADAPTER_DIR)
ADAPTER_DIR.mkdir(parents=True, exist_ok=True)

for epoch in range(1, EPOCHS + 1):
    adalora_model.train()
    classifier.train()
    running_loss = 0.0
    valid_micro_batches = 0
    skipped_batches = 0
    optimizer.zero_grad(set_to_none=True)

    pbar = tqdm(enumerate(train_loader, start=1), total=len(train_loader), desc=f"Epoch {epoch}/{EPOCHS}")
    for batch_idx, (ids, lens, labels) in pbar:
        ids    = ids.to(DEVICE, non_blocking=True)
        lens   = lens.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        with torch.amp.autocast("cuda", dtype=torch.float16):
            logits, _ = forward_document_batch(ids, lens)
            raw_loss = criterion(logits, labels)
            # AdaLoRA orthogonality penalty — PEFT would apply this inside its own
            # forward, but we bypass that forward, so we add it manually here.
            orth_reg = adalora_orth_reg(adalora_model, weight=0.1)
            loss = (raw_loss + orth_reg) / GRAD_ACCUM_STEPS

        if not torch.isfinite(loss):
            skipped_batches += 1
            optimizer.zero_grad(set_to_none=True)
            continue

        scaler.scale(loss).backward()
        running_loss += float(raw_loss.detach().cpu())
        valid_micro_batches += 1

        is_step_boundary = (batch_idx % GRAD_ACCUM_STEPS == 0) or (batch_idx == len(train_loader))
        if is_step_boundary:
            # Unscale and clip. grad_norm is the *pre-clip* total norm and will be
            # inf/nan on GradScaler overflow steps.
            scaler.unscale_(optimizer)
            grad_norm = torch.nn.utils.clip_grad_norm_(trainable_params, max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()

            if global_step < total_steps:
                scheduler.step()

            # AdaLoRA reads p.grad for importance scores, so this MUST run BEFORE
            # optimizer.zero_grad. On fp16-overflow steps (non-finite grad_norm)
            # the exponential moving averages would be permanently poisoned with
            # inf/nan, so we skip the importance update. Once the importance
            # phase has ended (global_step >= total_steps - TFINAL), grads are no
            # longer read, so it is safe to call even on overflow steps.
            if torch.isfinite(grad_norm) or global_step >= total_steps - TFINAL:
                adalora_model.base_model.update_and_allocate(global_step)

            optimizer.zero_grad(set_to_none=True)
            global_step += 1

        if batch_idx % 40 == 0:
            pbar.set_postfix(
                loss=f"{running_loss / max(1, valid_micro_batches):.4f}",
                lr=f"{scheduler.get_last_lr()[0]:.2e}",
                step=global_step,
            )

    mean_train_loss = float(running_loss / max(1, valid_micro_batches))
    epoch_losses.append({
        "epoch": int(epoch),
        "mean_train_loss": mean_train_loss,
        "skipped_batches": int(skipped_batches),
    })
    print(f"Epoch {epoch}: train_loss={mean_train_loss:.6f} | skipped_batches={skipped_batches}")

    # Save checkpoints at end of each epoch
    adalora_model.save_pretrained(str(ADAPTER_DIR), safe_serialization=True)
    torch.save(classifier.state_dict(), HEAD_PATH)
    torch.save(bb3.model.final_norm.state_dict(), FINAL_NORM_PATH)
    with open(OUT_DIR / "training_log.json", "w", encoding="utf-8") as f:
        json.dump({"epoch_losses": epoch_losses, "evaluation_performed": False}, f, indent=2)

train_seconds = time.time() - start_time
peak_gpu_gb = torch.cuda.max_memory_allocated() / 1024**3
print(f"Training completed in {train_seconds/60:.1f} min | Peak GPU memory={peak_gpu_gb:.2f} GiB")

# ======================= CLEAN RELOAD & TEST INFERENCE =======================
for _name in ("adalora_model", "bb3", "classifier", "optimizer", "scheduler", "scaler", "trainable_params", "adapter_params", "head_params"):
    globals().pop(_name, None)
torch.cuda.empty_cache()
gc.collect()

bb3 = load_backbone("arch3", ckpt_path=str(ARCH3_CKPT), device=str(DEVICE))
bb3.model.final_norm.load_state_dict(torch.load(FINAL_NORM_PATH, map_location=DEVICE, weights_only=True))
adalora_model = PeftModel.from_pretrained(bb3.model, str(ADAPTER_DIR), is_trainable=False)
adalora_model.eval()

classifier = EnhancedCJPEZoneClassifier(d_model=bb3.d_model).to(DEVICE)
classifier.load_state_dict(torch.load(HEAD_PATH, map_location=DEVICE, weights_only=True))
classifier.eval()

def arch3_hidden(ids):
    model = bb3.model
    B, T = ids.shape
    h = model.token_emb(ids) * model.embed_scale + model.pos_emb[:, :T, :].to(model.token_emb.weight.dtype)
    for block in model.blocks:
        h = block(h)
    return model.final_norm(h)

@torch.no_grad()
def predict_probs(ids_array, len_array, desc):
    ds = TensorDataset(torch.from_numpy(ids_array).long(), torch.from_numpy(len_array).long())
    loader = DataLoader(ds, batch_size=BATCH_DOCS * 2, shuffle=False, num_workers=2, pin_memory=True)
    probs, attention = [], []
    for ids, lens in tqdm(loader, desc=desc):
        ids  = ids.to(DEVICE, non_blocking=True)
        lens = lens.to(DEVICE, non_blocking=True)
        with torch.amp.autocast("cuda", dtype=torch.float16):
            logits, w = forward_document_batch(ids, lens)
        # Blend standard (T=1.0) and slightly sharpened (T=0.85) probabilities
        logits_f = logits.float()
        p1 = torch.softmax(logits_f, dim=-1)[:, 1]
        p2 = torch.softmax(logits_f / 0.85, dim=-1)[:, 1]
        p_blend = 0.5 * (p1 + p2)
        probs.extend(p_blend.cpu().numpy().tolist())
        attention.append(w.float().cpu().numpy())
    return np.asarray(probs), np.concatenate(attention, axis=0)

p_test, attn_test = predict_probs(X_test, L_test, "Test prediction (text only)")

# Prior-calibrated thresholding: aligns test positive prediction rate with the known
# training corpus positive rate (blended 70% quantile prior + 30% fixed 0.50 anchor for stability)
if CALIBRATE_THRESHOLD_TO_PRIOR:
    quantile_thr = float(np.quantile(p_test, 1.0 - TRAIN_POS_PRIOR))
    PREDICTION_THRESHOLD = float(np.clip(0.70 * quantile_thr + 0.30 * 0.50, 0.35, 0.65))
else:
    PREDICTION_THRESHOLD = 0.50

preds = (p_test >= PREDICTION_THRESHOLD).astype(np.int64)

submission = [{"id": str(row["id"]), "pred": int(pred)} for row, pred in zip(test_records, preds)]
assert len(submission) == len(test_records)
assert [x["id"] for x in submission] == test_ids
assert all(type(x["pred"]) is int and x["pred"] in (0, 1) for x in submission)

training_metadata = {
    "experiment": "Phase 4 Exp 2 Upgraded — Arch3 + High-Rank AdaLoRA + Focal Loss + 384-Len 4-Zone Fusion",
    "architecture": "Phase 1 Arch3 (242.5M parameters)",
    "adaptation": "AdaLoRA (init_r=16, target_r=8) + unfrozen final_norm",
    "original_train_examples": int(SOURCE_TRAIN_COUNT),
    "development_examples_merged_into_training": int(SOURCE_DEV_COUNT),
    "full_corpus_training_examples": int(len(X_train)),
    "test_examples": int(len(test_records)),
    "epochs": int(EPOCHS),
    "batch_docs_hardware": int(BATCH_DOCS),
    "grad_accum_steps": int(GRAD_ACCUM_STEPS),
    "effective_batch_size": int(BATCH_DOCS * GRAD_ACCUM_STEPS),
    "zone_length": int(MAX_ZONE_LEN),
    "train_positive_prior": float(TRAIN_POS_PRIOR),
    "prediction_threshold_calibrated": float(PREDICTION_THRESHOLD),
    "epoch_training_log": epoch_losses,
    "training_seconds": float(train_seconds),
    "peak_gpu_memory_gib": float(peak_gpu_gb),
    "mean_zone_attention_test": attn_test.mean(axis=0).tolist(),
    "test_positive_predictions": int((preds == 1).sum()),
    "test_negative_predictions": int((preds == 0).sum()),
    "test_labels_used": False,
    "evaluation_performed": False,
    "patches_applied": [
        "AdaLoRA update_and_allocate moved before optimizer.zero_grad",
        "AdaLoRA importance update skipped on fp16-overflow steps (isfinite guard)",
        "Manual orthogonal regularization reinstated (weight=0.1)",
    ],
}

with open(SUBMISSION_PATH, "w", encoding="utf-8") as f:
    json.dump(submission, f, ensure_ascii=False, indent=2)
with open(RESULTS_PATH, "w", encoding="utf-8") as f:
    json.dump(training_metadata, f, ensure_ascii=False, indent=2)
with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    zf.write(SUBMISSION_PATH, arcname="prediction.json")

print("\n" + "=" * 72)
print("UPGRADED PHASE 4 EXPERIMENT 2 — COMPLETED")
print("=" * 72)
print(f"Training time:              {train_seconds/60:.1f} minutes")
print(f"Peak allocated GPU memory:  {peak_gpu_gb:.2f} GiB")
print(f"Training Pos Prior:         {TRAIN_POS_PRIOR:.4f}")
print(f"Calibrated Threshold:       {PREDICTION_THRESHOLD:.4f}")
print("Test prediction counts:    ", Counter(int(x["pred"]) for x in submission))
print("Submission ZIP ready at:   ", ZIP_PATH)

PyTorch: 2.11.0+cu128
GPU: Tesla T4
GPU memory (GiB): 14.56
Speed & Context Profile: epochs=2, batch_docs=8 (eff=16), zone_len=384
Full labelled training corpus: 39,375 (Pos prior: 0.4143)
Test examples (id/text only):  1,517


Tokenizing full labelled corpus:   0%|          | 0/308 [00:00<?, ?it/s]

Tokenizing test inputs (text only):   0%|          | 0/12 [00:00<?, ?it/s]

Encoded shapes: (39375, 4, 384) (1517, 4, 384)
Mean zone lengths (train): [328.9, 329.1, 329.0, 329.1]
trainable params: 4,722,176 || all params: 247,221,856 || trainable%: 1.9101
Forward smoke test passed; logits shape: (1, 2)
Total optimizer steps=4,922 | tinit=393 | tfinal=738
Trainable params (AdaLoRA + final_norm + Head)=9,854,980 (3.905%)


Epoch 1/2:   0%|          | 0/4922 [00:00<?, ?it/s]

Epoch 1: train_loss=0.151681 | skipped_batches=0


Epoch 2/2:   0%|          | 0/4922 [00:00<?, ?it/s]

Epoch 2: train_loss=0.129624 | skipped_batches=0
Training completed in 291.7 min | Peak GPU memory=11.59 GiB


Test prediction (text only):   0%|          | 0/95 [00:00<?, ?it/s]


UPGRADED PHASE 4 EXPERIMENT 2 — COMPLETED
Training time:              291.7 minutes
Peak allocated GPU memory:  11.59 GiB
Training Pos Prior:         0.4143
Calibrated Threshold:       0.5112
Test prediction counts:     Counter({0: 850, 1: 667})
Submission ZIP ready at:    /kaggle/working/Phase4_Submissions/Exp2_Arch3_AdaLoRA_Improved/submission.zip
